# Hospital Readmission Risk - 30-Day Prediction
Operations aid for discharge planning. Target: readmitted <30 days (1) vs rest (0). Recall-first. Not a diagnosis.

## 1. Problem framing
Hospitals prioritize follow-up (calls, 7-day slots, pharmacist review). Metric: recall primary, ROC-AUC/PR-AUC secondary, precision for alert fatigue. Full plan in README.

In [ ]:
import pandas as pd, numpy as np, json, joblib
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv('data/diabetic_data.csv', na_values=['?'], low_memory=False)
print(df.shape)
print(df['readmitted'].value_counts())
print(df['readmitted'].value_counts(normalize=True).round(4))

## 2. Data understanding
101766 encounters, 71518 patients. Exclude expired/hospice discharges (11,13,14,19,20,21 = 2423 rows). Binary target ~11.4%.

In [ ]:
EXC=[11,13,14,19,20,21]
df2=df[~df['discharge_disposition_id'].isin(EXC)].copy()
df2['y']=(df2['readmitted']=='<30').astype(int)
print('after exclusion:',df2.shape,'pos_rate:',round(float(df2['y'].mean()),4))
print('missing pct top:')
print((df2.isna().mean().sort_values(ascending=False).head(8)*100).round(2))

## 3. EDA insights
Hypotheses: prior inpatient stays, polypharmacy, longer stay, A1C High, insulin/med changes, emergency admission raise risk.

In [ ]:
print('rate by prior inpatient:')
print(df2.groupby('number_inpatient')['y'].mean().head(8).round(3))
print('rate by A1C:')
print(df2.groupby('A1Cresult')['y'].mean().round(3))
print('rate by change:')
print(df2.groupby('change')['y'].mean().round(3))
df2.groupby('age')['y'].mean().plot(kind='bar',figsize=(9,3),title='30-day rate by age band')
plt.tight_layout(); plt.show()

## 4. Preprocessing (leakage-safe, see train.py)
Group-by-patient 80/20 split via StratifiedGroupKFold. ColumnTransformer inside Pipeline. ICD-9 grouping, med-change counts, Top-8 specialty grouping fit on train only.

## 5-6. Modeling, tuning, evaluation
Baseline + LogReg + RF + HGB, 5-fold GroupStratified CV scoring recall. Full run: python train.py. Results from metrics.json (own code).

In [ ]:
m=json.load(open('models/metrics.json'))
rows=[]
for k in ['baseline','logreg','rf','hgb']:
    v=m[k]
    rows.append({'model':k,'precision':v['precision'],'recall':v['recall'],'f1':v['f1'],'roc_auc':v['roc_auc'],'pr_auc':v['pr_auc']})
print(pd.DataFrame(rows).to_string(index=False))
print('Best: hgb ->',m['hgb'])

In [ ]:
from IPython.display import Image
Image('assets/roc.png')

In [ ]:
from IPython.display import Image
Image('assets/confusion.png')

## 7. Business insight
HGB catches 55.7 pct of 30-day returns at 18 pct precision. HIGH (>=0.50): 48h call + 7-day slot + pharmacist review. MEDIUM (0.20-0.50): 1-week call. LOW: standard care. Limits: 1999-2008, diabetes cohort, no adherence features.

## 8. App
Run: streamlit run app.py (loads models/model.joblib). Deploy to Streamlit Cloud / HF Spaces for public link.